# 01 — Monthly Data Audit

Audit the CRSP CIZ monthly production extract before any signal analysis.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.append(str(ROOT))
from src.clean_monthly import CleaningConfig, build_clean_monthly_panel
from src.concentration import calculate_monthly_concentration


## 1. Load raw production extract


In [ ]:
RAW = ROOT / 'data/raw/monthly_stock_common_equity_2000_2025_prod_v1.csv.gz'
raw = pd.read_csv(RAW, low_memory=False)
raw['MthCalDt'] = pd.to_datetime(raw['MthCalDt'])
print(f'Rows: {len(raw):,}')
print(f'Unique PERMNO: {raw.PERMNO.nunique():,}')
print(f'Months: {raw.MthCalDt.nunique():,}')
print(f'Date range: {raw.MthCalDt.min().date()} to {raw.MthCalDt.max().date()}')
raw.head()


## 2. Duplicate and missingness audit


In [ ]:
key = ['PERMNO', 'MthCalDt']
dup = raw[raw.duplicated(key, keep=False)]
print(f'Rows in duplicated keys: {len(dup):,}')
print(f'Exact full-row duplicates: {raw.duplicated().sum():,}')
missing = raw.isna().sum().to_frame('missing_n')
missing['missing_pct'] = 100 * missing['missing_n'] / len(raw)
missing.sort_values('missing_n', ascending=False).head(20)


## 3. Build the pre-specified clean panel


In [ ]:
config = CleaningConfig(min_price=5.0, nyse_size_percentile=0.20)
clean, audit = build_clean_monthly_panel(RAW, config)
audit


## 4. Concentration measures


In [ ]:
conc = calculate_monthly_concentration(clean)
conc.tail()


In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(conc['MthCalDt'], conc['Top10Share'])
ax.set_title('Top-10 market-cap share')
ax.set_xlabel('Month')
ax.set_ylabel('Top10Share')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(conc['MthCalDt'], conc['HHI'])
ax.set_title('Market-cap HHI')
ax.set_xlabel('Month')
ax.set_ylabel('HHI')
plt.show()
